# 4.6.1 Basic Autoencoders

Encoder-decoder MLP on California Housing: compress 8→3→8, then reconstruct.

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
relu=lambda x:np.maximum(0,x); relu_d=lambda x:(x>0).astype(float)

h=fetch_california_housing(); X=StandardScaler().fit_transform(h.data)
Xt,Xe=train_test_split(X,test_size=.2,random_state=42)
print('Train:',Xt.shape,'Test:',Xe.shape)

In [ ]:
rng=np.random.default_rng(42)
We1=rng.standard_normal((8,32))*np.sqrt(2/8); be1=np.zeros(32)
We2=rng.standard_normal((32,3))*np.sqrt(2/32); be2=np.zeros(3)
Wd1=rng.standard_normal((3,32))*np.sqrt(2/3); bd1=np.zeros(32)
Wd2=rng.standard_normal((32,8))*np.sqrt(2/32); bd2=np.zeros(8)
lr=.005; losses=[]
for _ in range(150):
    h1=relu(Xt@We1+be1); z=relu(h1@We2+be2); h2=relu(z@Wd1+bd1); xh=h2@Wd2+bd2
    loss=np.mean((xh-Xt)**2); losses.append(loss); n=len(Xt)
    dout=2*(xh-Xt)/n; Wd2-=lr*(h2.T@dout); bd2-=lr*dout.sum(0)
    dh2=(dout@Wd2.T)*relu_d(h2); Wd1-=lr*(z.T@dh2); bd1-=lr*dh2.sum(0)
    dz=(dh2@Wd1.T)*relu_d(z); We2-=lr*(h1.T@dz); be2-=lr*dz.sum(0)
    dh1=(dz@We2.T)*relu_d(h1); We1-=lr*(Xt.T@dh1); be1-=lr*dh1.sum(0)
print(f'Final MSE: {losses[-1]:.4f}')

In [ ]:
import matplotlib.pyplot as plt
xhat=relu(relu(relu(Xe@We1+be1)@We2+be2)@Wd1+bd1)@Wd2+bd2
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].plot(losses); axes[0].set_title('Training Loss')
axes[1].scatter(Xe[:200,0],xhat[:200,0],alpha=.4,s=8); axes[1].plot([-3,3],[-3,3],'r--')
axes[1].set_title('True vs Reconstructed (feature 0)'); plt.tight_layout(); plt.show()